# 07_04 Combining both: one model, two poems

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → [model learns] → model writes → MCP tool
> ```
>
> You are here: **model learns**
>
> Project files this connects to:
>
> - `src/homer_ai/train_homer.py`
> - `src/homer_ai/tokenizer.py`
> - `src/homer_ai/model.py`
>
> **This notebook:** Can ONE model learn both poems? We put `<|endoftext|>` between them, compare it fairly with the single-poem models, and say what it gives and costs.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to put both poems into one training text, with `<|endoftext|>` between them
- How to compare a combined model fairly with the two single-poem models
- How to read the result, and what a second change hidden in the test does
- What combining gives and what it costs

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **confound** | a second change hidden inside the one you meant to test, so you can't tell which one caused the result | the combined model also saw twice as much text |
| **specialist** | a model trained on only one poem, as opposed to one trained on both | the Odyssey-only model is the specialist on the Odyssey |

## 3. The idea

### ❓ How do I make ONE training text from two poems?

With the same loop as `load_data` in `train_homer.py` ([02_02](../02_text_to_numbers/02_02_special_tokens.ipynb)). We encode each text on its own, and put the number of `<|endoftext|>` between them. The tokenizer is the shared vocabulary of [07_03](07_03_comparing_fairly.ipynb). The marker is how the model learns that one work has ended and another begins.

```
Odyssey slice → [ … ]  +  [<|endoftext|>]  +  Iliad slice → [ … ]
```

👉 **Predict first:** the code prints the end of the Odyssey slice, the marker, and the start of the Iliad slice. What will it show?

In [ ]:
import torch

from homer_ai.lessons import HOMER, evaluate, fit, load_book_files, split_90_10
from homer_ai.tokenizer import CharTokenizer

SLICE = 10000  # characters of Book 1 we use: small, so every cell runs in seconds

odyssey = load_book_files(HOMER / "odyssey" / "greek")  # 24 books, one text each
iliad = load_book_files(HOMER / "iliad" / "greek")

In [ ]:
shared = CharTokenizer("\n".join(odyssey + iliad))  # the shared vocabulary (07_03)


def join_with_eot(parts):  # the loop of load_data (02_02); parts are tensors
    ids = []
    for part in parts:
        if ids:  # not the first part: mark where the previous one ended
            ids.append(shared.eot_id)
        ids.extend(part.tolist())
    return torch.tensor(ids, dtype=torch.long)


odyssey_1 = torch.tensor(shared.encode(odyssey[0][:SLICE]))  # slice of Book 1
iliad_1 = torch.tensor(shared.encode(iliad[0][:SLICE]))
odyssey_train, odyssey_val = split_90_10(odyssey_1)
iliad_train, iliad_val = split_90_10(iliad_1)

both = join_with_eot([odyssey_train, iliad_train])
at = len(odyssey_train)  # where the Odyssey part ends
print("length:", len(both), "=", len(odyssey_train), "+ 1 marker +", len(iliad_train))
print("ends of the Odyssey part:", repr(shared.decode(both[at - 8 : at].tolist())))
marker = repr(shared.decode(both[at : at + 1].tolist()))
print("the marker:              ", marker, "= number", both[at].item())
print("start of the Iliad part: ", repr(shared.decode(both[at + 1 : at + 9].tolist())))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The combined text is the two training parts plus one marker (number 174), with the Odyssey's last 8 characters on one side and the Iliad's first 8 on the other. The marker is one token, not 13 characters, just as in [02_02](../02_text_to_numbers/02_02_special_tokens.ipynb).

</details>

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: what exactly do I compare?

Four models. They all get the shared vocabulary, 4 epochs and seeds 0 and 1, and they're all tested on both held-out parts:

| Model | Trained on | Training characters |
|---|---|---|
| A | Odyssey slice | 9,000 |
| B | Iliad slice | 9,000 |
| C | both slices, in full | 18,001 |
| D | the first half of each slice | 9,001 |

C is the natural "combined" model, but it sees twice as much text as A and B, so it changes two things at once: the mix and the amount. That's a **confound**, and it breaks the rule of [05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb) (change one thing). D fixes it. It gets the same amount of text as A and B (9,001 characters), half from each poem. So comparing D with A and B changes only the mix.

👉 **Predict first:** which will be best on the Odyssey's held-out part, A or C or D? And what will D look like next to A and B?

In [ ]:
SEEDS = (0, 1)
half = len(odyssey_train) // 2
training = {
    "A  Odyssey": odyssey_train,
    "B  Iliad": iliad_train,
    "C  both, in full": join_with_eot([odyssey_train, iliad_train]),
    "D  both, half each": join_with_eot([odyssey_train[:half], iliad_train[:half]]),
}

results = {}  # name -> (loss on Odyssey held-out, loss on Iliad held-out)
first_runs = {}  # name -> the seed-0 model, kept for writing below
for name, data in training.items():
    runs = [fit(data, shared.vocab_size, seed) for seed in SEEDS]
    first_runs[name] = runs[0]
    results[name] = (
        sum(evaluate(m, odyssey_val) for m in runs) / len(runs),
        sum(evaluate(m, iliad_val) for m in runs) / len(runs),
    )

print(f"{'model':20} {'chars':>6} {'Odyssey held-out':>17} {'Iliad held-out':>15}")
for name, (on_odyssey, on_iliad) in results.items():
    print(f"{name:20} {len(training[name]):>6,} {on_odyssey:>17.3f} {on_iliad:>15.3f}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

C (both poems, in full) matches A on the Odyssey (2.800 against 2.811) and beats B on the Iliad (2.809 against 2.873). D (the same amount of text as A and B, half from each poem) is **worse than the specialist on each poem**. On the Odyssey it gets 2.930 against A's 2.811, and on the Iliad 3.024 against B's 2.873. In fact on the Iliad, D (3.024) is no better than A (3.026), which never saw the Iliad. The graph shows it. Just note that its axis starts at 2.5, which makes small gaps look big.

</details>

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import (
    ACCENT,
    BAD,
    GOOD,
    INK,
    MUTED,
    studio_axes,
    studio_figure,
)

title = "Held-out loss of the four models (lower is better)"
fig = studio_figure("combining", title, (10, 6))
ax = studio_axes(fig, (0.1, 0.12, 0.8, 0.64))
colours = [ACCENT, GOOD, BAD, MUTED]
width = 0.2
for n, (name, colour) in enumerate(zip(results, colours, strict=True)):
    positions = [i + (n - 1.5) * width for i in range(2)]
    ax.bar(positions, results[name], width, color=colour, label=name)
    for x, value in zip(positions, results[name], strict=True):
        ax.text(x, value + 0.01, f"{value:.2f}", ha="center", color=INK,
                family="monospace", fontsize=10, fontweight="bold")
ax.set_xticks([0, 1], ["Odyssey held-out", "Iliad held-out"])
ax.tick_params(axis="x", labelsize=13, labelcolor=INK)
ax.set_ylim(2.5, 3.25)
ax.set_ylabel("loss (this axis starts at 2.5, not 0)")
ax.grid(axis="x", visible=False)
ax.set_axisbelow(True)
ax.legend(facecolor="none", edgecolor=MUTED, labelcolor=INK, loc="upper center",
          ncols=2)
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

C looks best, but it isn't a fair win. It saw twice as much text, so it also took twice as many training steps. D changes **only the mix**, and it's worse than each specialist. So in this experiment, mixing the poems didn't help by itself, and C's gain came from more text. Here's a guess, which we didn't test: the model is small, with only 14,272 weights (`175 × 32 × 2` for the embedding and the head, plus `3 × 32 × 32` for query, key and value), and they have to serve two sets of habits. We have two seeds and tiny slices, so a bigger model, more text or more epochs could change this.

</details>

### ❓ Step 2: does the combined model answer differently to different prompts?

We can ask a model that knows both poems with the first word of each: `μῆνιν` (the Iliad's "anger") and `ἄνδρα` (the Odyssey's "man"). We use model C with top-k 5 and temperature 0.8 ([06_02](../06_making_it_talk/06_02_less_repetition.ipynb)), the same seed, and 70 new characters. Model A (Odyssey only) and B (Iliad only) get the same prompts for comparison.

👉 **Predict first:** will the combined model's text start to look like the poem the prompt came from?

In [ ]:
from homer_ai.lessons import write  # the function of 06_02

for model_name in ("A  Odyssey", "B  Iliad", "C  both, in full"):
    for prompt in ("μῆνιν", "ἄνδρα"):
        text = write(first_runs[model_name], shared, prompt, 70)
        print(f"{model_name[0]} {prompt:7}", text)
    print()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The prompt only changes the first few characters. After `μῆνιν` or `ἄνδρα`, each model goes on with almost the **same text** for both prompts (model C writes `…τενα με τὰ κει τεν τʼ ἐγο καν καν δʼ …` both times). The prompt is only 5 characters, and it stays inside the model's 24-character window until 19 more characters are written. Even so, the two texts are almost the same. For this small model the prompt hardly steers the writing. Models B and C, which saw the Iliad, begin alike (`μῆνινα τε…`), while A, the Odyssey-only model, writes something else (`μῆνιντουν ἀλοὐ…`). So we see no sign that the combined model writes Iliad-like Greek after `μῆνιν` and Odyssey-like Greek after `ἄνδρα`. To be fair, expecting a model this small to separate the two poems would be asking too much.

</details>

### ❓ So what does combining give, and what does it cost?

Going by what we measured:

**What it gives**

- One model, one file and one tool for both poems.
- More text to learn from. C beat B on the Iliad's held-out part (2.809 against 2.873) and matched A on the Odyssey's.
- One vocabulary, so every character of either poem is known.

**What it costs**

- With the same amount of text, the mixed model D was worse than each specialist on its own poem (2.930 against 2.811, and 3.024 against 2.873).
- The model's few weights must cover two poems, and two editors' habits ([07_01](07_01_the_corpus.ipynb)).
- It can't say which poem it's writing. The prompt didn't steer it (above), so you can't simply ask for "Iliad Greek".
- Twice the text means twice the training time.

So for a model this small, combining is a convenience. It doesn't make the model better. It gets interesting with a bigger model (the parts of chapter 4) and the whole texts.

### ❓ What about the other two ways to combine?

**Two models side by side** needs no new maths. You train A and B as before, and a program (like the MCP server of [06_03](../06_making_it_talk/06_03_the_mcp_server.ipynb)) offers one tool for each, or one tool with a `poem` argument that picks the model. Each model keeps its own poem's habits, and you pay with two files and two training runs.

**Fine-tuning** (training on the Odyssey, then a little more on the Iliad) isn't built here. It needs care with the learning rate so the model doesn't forget the first poem, and our tiny model would show little. It's the idea behind adapting big models to a new job, and [chapter 8](../08_advanced_fine_tuning/08_00_chapter_overview.ipynb) does exactly that with GPT-2.

## 5. Recap

- For one model on both poems, encode each text, put `<|endoftext|>` between them, and use the shared vocabulary.
- On the same amount of text, the mixed model was worse than each specialist on its own poem (2.930 against 2.811, and 3.024 against 2.873). The combined model's gain came from seeing more text.
- Combining means one file and one tool, but the model's few weights now serve two styles, and the prompt didn't steer it towards one poem or the other.

**Next:** [07_05: English and Greek](07_05_english_and_greek.ipynb).